# METR-LA — Naive / GRU / GCN / GRU-GCN
1. Missing value (ค่า 0) ถูก mask ทั้งใน loss และ metric, input เติมด้วย forward-fill
2. Scaler คำนวณจากค่าที่วัดได้จริงใน train เท่านั้น
3. ทุกโมเดลเป็น PyTorch, เทรนด้วย loop เดียวกัน, ตั้ง seed, early stopping
4. ทุกโมเดล predict แบบ residual (ค่าล่าสุด + delta) และ GCN มี self/skip path, ไม่เกิด oversmoothing
5. Metric ทุกตัวคิดเป็น mph (inverse transform แล้ว) เปรียบเทียบกันได้
6. แบ่ง train/val/test ตามเวลาของ target (ไม่เสีย window ตอนต้น val/test)

In [ ]:
import os, pickle, random, copy
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from torch.utils.data import TensorDataset, DataLoader

SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", device)  # ถ้าเป็น cpu: Runtime > Change runtime type > GPU

Device: cpu


## 1) โหลดข้อมูล

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

folder_path = '/content/drive/MyDrive/My_Project/GraphTraffic-AI/TrafficFlow'

df = pd.read_hdf(os.path.join(folder_path, 'METR-LA.h5'))
with open(os.path.join(folder_path, 'adj_METR-LA.pkl'), 'rb') as f:
    sensor_ids, sensor_id_to_ind, adj_mx = pickle.load(f, encoding='latin1')

assert list(df.columns) == list(sensor_ids), "ลำดับ sensor ไม่ตรงกับ adjacency!"
print("Data:", df.shape, "| Adj:", adj_mx.shape)
print("Missing (value == 0): %.2f%%" % (100 * (df.values == 0).mean()))

Mounted at /content/drive
Data: (34272, 207) | Adj: (207, 207)
Missing (value == 0): 8.11%


## 2) เตรียมข้อมูล (mask missing, normalize, window)

In [ ]:
WINDOW_IN, WINDOW_OUT = 12, 1
n = len(df)
train_end, val_end = int(n * 0.70), int(n * 0.80)

raw = df.values.astype(np.float32)
obs = raw != 0                                   # True = วัดได้จริง

# scaler: ใช้เฉพาะค่าที่วัดได้ใน train
train_vals = raw[:train_end][obs[:train_end]]
mean, std = float(train_vals.mean()), float(train_vals.std())
print(f"Train mean={mean:.3f}  std={std:.3f}  (เฉพาะค่าที่วัดได้)")

# input: forward-fill ค่าที่ขาด (ใช้เฉพาะอดีต) แล้ว normalize
filled = df.where(df != 0).ffill().fillna(mean).values.astype(np.float32)
filled_norm = (filled - mean) / std
# target: ค่าจริง normalize + mask
target_norm = np.where(obs, (raw - mean) / std, 0.0).astype(np.float32)

def make_windows(t_start, t_end):
    # window ถูกกำหนดด้วยเวลาของ target t; input = 12 step ก่อนหน้า t
    t = np.arange(max(t_start, WINDOW_IN), t_end)
    idx = t[:, None] + np.arange(-WINDOW_IN, 0)[None, :]      # (B, T)
    X = np.ascontiguousarray(filled_norm[idx].transpose(0, 2, 1))  # (B, N, T)
    return X, target_norm[t], obs[t].astype(np.float32)

X_train, Y_train, M_train = make_windows(0, train_end)
X_val,   Y_val,   M_val   = make_windows(train_end, val_end)
X_test,  Y_test,  M_test  = make_windows(val_end, n)

for name, a in [("train", X_train), ("val", X_val), ("test", X_test)]:
    print(name, a.shape)
print("Target observed ratio (test): %.3f" % M_test.mean())

Train mean=58.584  std=12.823  (เฉพาะค่าที่วัดได้)
train (23978, 207, 12)
val (3427, 207, 12)
test (6855, 207, 12)
Target observed ratio (test): 0.879


## 3) Graph (adjacency แบบ row-normalized)

In [ ]:
A = adj_mx.astype(np.float32).copy()
np.fill_diagonal(A, 1.0)                 # ตั้ง self-loop = 1 (ไม่บวกซ้ำ)
A = A / A.sum(axis=1, keepdims=True)     # random-walk normalization D^-1 A
adj = torch.tensor(A, device=device)

print("Row sums (min/max):", A.sum(1).min(), A.sum(1).max())
print("Non-zero:", np.count_nonzero(A), "| Density: %.3f" % (np.count_nonzero(A) / A.size))

def to_loader(X, Y, M, shuffle):
    ds = TensorDataset(torch.from_numpy(X), torch.from_numpy(Y), torch.from_numpy(M))
    return DataLoader(ds, batch_size=64, shuffle=shuffle)

train_loader = to_loader(X_train, Y_train, M_train, True)
val_loader   = to_loader(X_val,   Y_val,   M_val,   False)
test_loader  = to_loader(X_test,  Y_test,  M_test,  False)

Row sums (min/max): 0.9999999 1.0000001
Non-zero: 1722 | Density: 0.040


## 4) Metric (mph, masked) + Naive baseline

In [ ]:
def masked_metrics(pred_norm, true_norm, mask):
    pred = pred_norm * std + mean
    true = true_norm * std + mean
    err = (pred - true) * mask
    denom = mask.sum()
    return float(np.abs(err).sum() / denom), float(np.sqrt((err ** 2).sum() / denom))

results = {}

# Naive: ใช้ค่าล่าสุด (ที่ forward-fill แล้ว)
mae, rmse = masked_metrics(X_test[:, :, -1], Y_test, M_test)
results["Naive"] = (mae, rmse)
print(f"Naive  | MAE {mae:.4f} | RMSE {rmse:.4f}  (mph, masked)")

Naive  | MAE 2.8132 | RMSE 4.6250  (mph, masked)


## 5) โมเดล

In [ ]:
class GRUMultivariate(nn.Module):
    # เหมือน GRU เดิมใน Keras: ใส่ทั้ง 207 sensor เป็น feature ของ GRU ตัวเดียว
    def __init__(self, n_nodes=207, hidden=64):
        super().__init__()
        self.gru = nn.GRU(n_nodes, hidden, batch_first=True)
        self.head = nn.Linear(hidden, n_nodes)
        nn.init.zeros_(self.head.weight); nn.init.zeros_(self.head.bias)

    def forward(self, x, adj=None):               # x: (B, N, T)
        out, _ = self.gru(x.transpose(1, 2))       # (B, T, H)
        return x[:, :, -1] + self.head(out[:, -1])


class GraphLayer(nn.Module):
    # relu( W_n * (A h) + W_s * h )  -> มี self path กัน oversmoothing
    def __init__(self, d_in, d_out):
        super().__init__()
        self.neigh = nn.Linear(d_in, d_out)
        self.self_ = nn.Linear(d_in, d_out, bias=False)

    def forward(self, h, adj):                     # h: (B, N, D)
        return torch.relu(self.neigh(torch.matmul(adj, h)) + self.self_(h))


class GCN(nn.Module):
    def __init__(self, t_in=12, hidden=64):
        super().__init__()
        self.g1 = GraphLayer(t_in, hidden)
        self.g2 = GraphLayer(hidden, hidden)
        self.head = nn.Linear(hidden, 1)
        nn.init.zeros_(self.head.weight); nn.init.zeros_(self.head.bias)

    def forward(self, x, adj):                     # x: (B, N, T)
        h = self.g2(self.g1(x, adj), adj)
        return x[:, :, -1] + self.head(h).squeeze(-1)


class GRUGCN(nn.Module):
    def __init__(self, hidden=64):
        super().__init__()
        self.gru = nn.GRU(1, hidden, batch_first=True)   # ใช้ร่วมกันทุก node
        self.g1 = GraphLayer(hidden, hidden)
        self.g2 = GraphLayer(hidden, hidden)
        self.head = nn.Linear(hidden, 1)
        nn.init.zeros_(self.head.weight); nn.init.zeros_(self.head.bias)

    def forward(self, x, adj):                     # x: (B, N, T)
        B, N, T = x.shape
        _, h = self.gru(x.reshape(B * N, T, 1))    # h: (1, B*N, H)
        h = h[-1].reshape(B, N, -1)
        h = self.g2(self.g1(h, adj), adj)
        return x[:, :, -1] + self.head(h).squeeze(-1)

## 6) Training utilities

In [ ]:
def masked_l1(pred, y, m):
    return (torch.abs(pred - y) * m).sum() / m.sum().clamp(min=1.0)

@torch.no_grad()
def predict(model, loader):
    model.eval()
    P, Y, M = [], [], []
    for x, y, m in loader:
        P.append(model(x.to(device), adj).cpu().numpy())
        Y.append(y.numpy()); M.append(m.numpy())
    return np.concatenate(P), np.concatenate(Y), np.concatenate(M)

def train(model, epochs=30, lr=1e-3, patience=5):
    model.to(device)
    opt = torch.optim.Adam(model.parameters(), lr=lr)
    best, best_state, bad = float("inf"), None, 0
    for ep in range(1, epochs + 1):
        model.train()
        tl = 0.0
        for x, y, m in train_loader:
            x, y, m = x.to(device), y.to(device), m.to(device)
            opt.zero_grad()
            loss = masked_l1(model(x, adj), y, m)
            loss.backward()
            nn.utils.clip_grad_norm_(model.parameters(), 5.0)
            opt.step()
            tl += loss.item()
        val_mae, val_rmse = masked_metrics(*predict(model, val_loader))
        print(f"Epoch {ep:02d} | train L1(norm) {tl/len(train_loader):.4f} "
              f"| val MAE {val_mae:.3f} mph | val RMSE {val_rmse:.3f}")
        if val_mae < best - 1e-4:
            best, bad = val_mae, 0
            best_state = copy.deepcopy(model.state_dict())
        else:
            bad += 1
            if bad >= patience:
                print("Early stop"); break
    model.load_state_dict(best_state)
    return model

def evaluate(name, model):
    mae, rmse = masked_metrics(*predict(model, test_loader))
    results[name] = (mae, rmse)
    print(f"{name} | TEST MAE {mae:.4f} | RMSE {rmse:.4f} (mph, masked)")

## 7) Sanity check — overfit 1 batch
ก่อนเทรนจริง ให้โมเดลเรียน 1 batch ซ้ำๆ loss ต้องลดลงชัดเจน ถ้าไม่ลด แปลว่าโค้ดโมเดล/optimizer มีปัญหา

In [ ]:
def overfit_check(model_fn, steps=300, lr=3e-3):
    torch.manual_seed(SEED)
    model = model_fn().to(device)
    opt = torch.optim.Adam(model.parameters(), lr=lr)
    x, y, m = [t.to(device) for t in next(iter(train_loader))]
    for s in range(steps):
        opt.zero_grad()
        loss = masked_l1(model(x, adj), y, m)
        loss.backward(); opt.step()
        if s in (0, steps // 2, steps - 1):
            print(f"  step {s:3d} | loss {loss.item():.4f}")

for name, fn in [("GRU", GRUMultivariate), ("GCN", GCN), ("GRU-GCN", GRUGCN)]:
    print(name); overfit_check(fn)

GRU
  step   0 | loss 0.2282
  step 150 | loss 0.0171
  step 299 | loss 0.0064
GCN
  step   0 | loss 0.2064
  step 150 | loss 0.1676
  step 299 | loss 0.1597
GRU-GCN
  step   0 | loss 0.2166
  step 150 | loss 0.1796
  step 299 | loss 0.1700


## 8) เทรนจริง

In [ ]:
torch.manual_seed(SEED)
print("=== GRU ===");     gru_model = train(GRUMultivariate());  evaluate("GRU", gru_model)

=== GRU ===
Epoch 01 | train L1(norm) 0.2100 | val MAE 2.578 mph | val RMSE 4.277
Epoch 02 | train L1(norm) 0.2054 | val MAE 2.550 mph | val RMSE 4.230
Epoch 03 | train L1(norm) 0.2031 | val MAE 2.535 mph | val RMSE 4.206
Epoch 04 | train L1(norm) 0.2017 | val MAE 2.524 mph | val RMSE 4.190
Epoch 05 | train L1(norm) 0.2006 | val MAE 2.515 mph | val RMSE 4.180
Epoch 06 | train L1(norm) 0.1999 | val MAE 2.511 mph | val RMSE 4.173
Epoch 07 | train L1(norm) 0.1993 | val MAE 2.507 mph | val RMSE 4.166
Epoch 08 | train L1(norm) 0.1989 | val MAE 2.504 mph | val RMSE 4.163
Epoch 09 | train L1(norm) 0.1985 | val MAE 2.502 mph | val RMSE 4.159
Epoch 10 | train L1(norm) 0.1981 | val MAE 2.497 mph | val RMSE 4.153
Epoch 11 | train L1(norm) 0.1978 | val MAE 2.497 mph | val RMSE 4.154
Epoch 12 | train L1(norm) 0.1976 | val MAE 2.494 mph | val RMSE 4.149
Epoch 13 | train L1(norm) 0.1973 | val MAE 2.491 mph | val RMSE 4.147
Epoch 14 | train L1(norm) 0.1971 | val MAE 2.489 mph | val RMSE 4.145
Epoch 15

In [ ]:
torch.manual_seed(SEED)
print("=== GCN ===");     gcn_model = train(GCN());              evaluate("GCN", gcn_model)

=== GCN ===
Epoch 01 | train L1(norm) 0.1879 | val MAE 2.250 mph | val RMSE 3.968
Epoch 02 | train L1(norm) 0.1814 | val MAE 2.223 mph | val RMSE 3.922
Epoch 03 | train L1(norm) 0.1800 | val MAE 2.214 mph | val RMSE 3.908
Epoch 04 | train L1(norm) 0.1794 | val MAE 2.210 mph | val RMSE 3.890
Epoch 05 | train L1(norm) 0.1789 | val MAE 2.204 mph | val RMSE 3.881
Epoch 06 | train L1(norm) 0.1786 | val MAE 2.204 mph | val RMSE 3.868
Epoch 07 | train L1(norm) 0.1783 | val MAE 2.196 mph | val RMSE 3.870
Epoch 08 | train L1(norm) 0.1780 | val MAE 2.196 mph | val RMSE 3.873
Epoch 09 | train L1(norm) 0.1778 | val MAE 2.192 mph | val RMSE 3.853
Epoch 10 | train L1(norm) 0.1775 | val MAE 2.188 mph | val RMSE 3.852
Epoch 11 | train L1(norm) 0.1774 | val MAE 2.187 mph | val RMSE 3.842
Epoch 12 | train L1(norm) 0.1773 | val MAE 2.187 mph | val RMSE 3.845
Epoch 13 | train L1(norm) 0.1771 | val MAE 2.186 mph | val RMSE 3.857
Epoch 14 | train L1(norm) 0.1770 | val MAE 2.185 mph | val RMSE 3.843
Epoch 15

In [ ]:
torch.manual_seed(SEED)
print("=== GRU-GCN ==="); grugcn_model = train(GRUGCN());        evaluate("GRU-GCN", grugcn_model)

=== GRU-GCN ===
Epoch 01 | train L1(norm) 0.1872 | val MAE 2.240 mph | val RMSE 3.928
Epoch 02 | train L1(norm) 0.1801 | val MAE 2.210 mph | val RMSE 3.883
Epoch 03 | train L1(norm) 0.1784 | val MAE 2.193 mph | val RMSE 3.863
Epoch 04 | train L1(norm) 0.1774 | val MAE 2.186 mph | val RMSE 3.852
Epoch 05 | train L1(norm) 0.1767 | val MAE 2.178 mph | val RMSE 3.827
Epoch 06 | train L1(norm) 0.1762 | val MAE 2.178 mph | val RMSE 3.828
Epoch 07 | train L1(norm) 0.1759 | val MAE 2.169 mph | val RMSE 3.808
Epoch 08 | train L1(norm) 0.1756 | val MAE 2.170 mph | val RMSE 3.812
Epoch 09 | train L1(norm) 0.1754 | val MAE 2.174 mph | val RMSE 3.792
Epoch 10 | train L1(norm) 0.1752 | val MAE 2.172 mph | val RMSE 3.802
Epoch 11 | train L1(norm) 0.1751 | val MAE 2.161 mph | val RMSE 3.800
Epoch 12 | train L1(norm) 0.1750 | val MAE 2.158 mph | val RMSE 3.794
Epoch 13 | train L1(norm) 0.1748 | val MAE 2.161 mph | val RMSE 3.801
Epoch 14 | train L1(norm) 0.1746 | val MAE 2.158 mph | val RMSE 3.796
Epoc

## 9) สรุปผล (mph, masked, test set)

In [ ]:
summary = pd.DataFrame(results, index=["MAE", "RMSE"]).T.round(4)
display(summary)

,MAE,RMSE
Naive,2.8132,4.6250
GRU-GCN,2.3079,4.0081
